# Problem Set 2: Agents on the registry and the compliance checker
### Chapter 3 — revisiting Chapters 1 and 2

**Topic coverage:** the agent layer of Chapter 3 (`agentlab.tools`, `agentlab.agents`), applied to the tasks of Chapter 1 (the ontology registry, Keet §1.2–1.3) and Chapter 2 (formalised policy, closed-domain entailment, Keet §2.2).

**Level:** Graduate (practice-first).

**Total points:** 30 (Part A 8 / Part B 10 / Part C 12); 13 pts are auto-graded by the check cells, the rest are marked by hand against the rubric. **Estimated effort:** 4–5 hours.

**API budget:** ≈ \$0.50–1.50 **estimated, not yet measured** on `claude-sonnet-5-5`: 14 agent episodes (a few cents each) and 8 DSPy calls. Part A calls no model.

## Why this problem set exists

Chapters 1 and 2 built their tasks before the course had tools or agents, so their problem sets stopped where an agent was needed. Those parts return here, on the original tasks and data:

* **Part A** (original Chapter 1, A2): the registry as narrow, logged tools — and what a one-call "mega-tool" hides.
* **Part B** (original Chapter 1, D1): one agent loop against a fixed program with a verifying critic, on Meridian Rail's intake triage.
* **Part C** (original Chapter 2, D1): a compliance agent that proves its verdicts with an entailment checker, on St. Brigid's access policy.

The support code is `ch03_revisit.py`. It imports Chapter 1's registry corpus and Chapter 2's FOL engine: a revisit problem set may reach *back* to an earlier chapter, never ahead. Chapter 1's policy functions (its A1 answer) are provided as `R.reportable_smells` and `R.intake_decision`.

## Submission requirements

1. **This notebook, executed top to bottom**, with every `TODO` implemented and every *Your answer* cell written.
2. **Honest reporting.** A number without its sample size and its cost earns no credit.

The check cells are the marking scheme for the auto-graded points; do not edit them. The final self-test prints your auto-graded subtotal.

> Solutions: `06_solutions.ipynb` — do not open it until you have attempted every problem.

In [ ]:
# revisits: ch01_introduction, ch02_first_order_logic
import sys, json, logging
from pathlib import Path

# Make the course root (the folder holding agentlab/) and this chapter importable.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "agentlab").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import agentlab
from agentlab import llm
from agentlab.grading import Grader

print(json.dumps(agentlab.describe_environment(), indent=1))
llm.require_credentials()             # this problem set calls the live Anthropic API
logging.getLogger("dspy").setLevel(logging.WARNING)
GRADER = Grader('Chapter 3 — Problem Set 2 — Agents on the registry and the compliance checker')

import re
import dspy
import pandas as pd
from agentlab import evaluation as ev, agents
import ch03_revisit as R
import ch01_agentic as A          # Chapter 1: the registry corpus
import ch02_toolkit as fol        # Chapter 2: the FOL engine

ALL, test = A.build_dataset(), A.build_dataset("test")
reportable_smells, intake_decision = R.reportable_smells, R.intake_decision
print(f"{len(ALL)} registry submissions ({len(test)} in test); "
      f"{len(R.COMPLIANCE_QUERIES)} compliance questions")

---

## Part A — The registry as tools (original Chapter 1, A2)

### A2. Tools, trajectories, and what a mega-tool hides (8 pts)

The registry tools (`R.build_registry_tools(ws)`) are narrow and logged: every call lands
in `ws.log` (a `ToolCallLog`). `R.TOOL_TO_EVIDENCE` names the kind of evidence each tool
buys.

1. Write `evidence_from_log(log) -> frozenset` — the evidence kinds a trajectory bought.
2. Write `scripted_review(submission_id, ws) -> dict` with keys `level`, `smells`,
   `decision`: a *deterministic reviewer* that invokes (with `.invoke`) only the registry
   tools it needs and applies your A1 functions. Run it on all 24 submissions, each in a
   fresh workspace, and set `control_accuracy` to the fraction it gets fully right.
3. Build `a2`, one row per toolset for the submission `"interlocking-import"`: the
   decomposed trajectory of your scripted reviewer and one call of the monolithic
   `R.build_monolithic_tool(ws)`; columns `toolset` (`"decomposed"` / `"monolithic"`),
   `calls`, `result_chars` (sum over the log), `evidence_visible`
   (`sorted(evidence_from_log(...))`).

In writing: what does the monolithic tool cost you — think of the trajectory as a record,
of reward shaping, and of least privilege — and when would you still ship one? Read the
tool descriptions in `ch01_agentic.py`: which sentence in `submission_record`'s description
makes an agent call it, and what would happen if it were missing?

#### Background and design decisions — A2

This is where you meet the **workspace** and the **tool registry** (Background §2–4) as
a user. Build the tools on a workspace, call them with `.invoke`, decode the JSON, and
look at `ws.log`. The scripted reviewer shows what the task costs *without any model*: if
code gets 24/24 with four calls, every Claude result later is measured against that.

Decisions you make:

* **Which tools to call.** Only what the policy needs: the record (declared level), the
  spectrum (measured level), the scanner. `graph_metrics` is not needed to decide; every
  extra call is cost and noise in the trajectory.
* **A fresh workspace per submission**, so each log is one clean trajectory and nothing
  from the previous item leaks into the next.
* **What counts as evidence.** `evidence_from_log` counts only successful calls: a failed
  call bought nothing. `load_submission` buys no evidence; it only makes the others
  possible.
* **Narrow vs monolithic.** The monolithic tool is *one* call that returns everything,
  so the log cannot say what the decision used. The written part asks when you would still
  ship one.

#### Contract — A2

```python
def evidence_from_log(log: ToolCallLog) -> frozenset[str]
def scripted_review(submission_id: str, ws: R.RegistryWorkspace) -> dict
control_accuracy: float
a2: pd.DataFrame
```

**`evidence_from_log(log)`** — `log` is a workspace's `ws.log`. Loop over `log.calls`;
for every call with `ok == True` whose `name` is a key of `R.TOOL_TO_EVIDENCE`, collect
`R.TOOL_TO_EVIDENCE[name]`. Return the kinds as a `frozenset[str]` (a subset of
`R.EVIDENCE_KINDS`). Example: a log of `load_submission, spectrum_position, scan_smells`
→ `frozenset({"spectrum", "smells"})`.

**`scripted_review(submission_id, ws)`** — a deterministic reviewer, no model.

| parameter | type | meaning |
|---|---|---|
| `submission_id` | `str` | e.g. `"signalling-equipment"` |
| `ws` | `R.RegistryWorkspace` | a **fresh** workspace; build the tools on it so every call lands in `ws.log` |

It must call `load_submission` first, then only the tools whose output it needs
(`submission_record` for the declared level, `spectrum_position`, `scan_smells`), **at
most 5 calls in total, none failing**. Returns a dict with **exactly** these three keys
(the check compares the whole dict with `==`):

```python
{"level": "taxonomy",                                   # str, the measured level
 "smells": ["no-disjointness", "subsumption-cycle"],    # reportable_smells(...) output
 "decision": "reject"}                                  # intake_decision(...) output
```

Remember that `scan_smells` returns one entry per finding, so collect the `"smell"` values
into a set before calling `reportable_smells`.

**`control_accuracy`** — `float` in [0, 1]: over all 24 rows of `ALL`, the fraction for
which `scripted_review(ex.id, R.RegistryWorkspace()) == {"level": ex.level, "smells":
ex.smells, "decision": ex.decision}`.

**`a2`** — a 2-row DataFrame for submission `"interlocking-import"`:

| column | type | meaning |
|---|---|---|
| `toolset` | `str` | `"decomposed"` (your `scripted_review`) or `"monolithic"` (one `review_submission` call) |
| `calls` | `int` | `len(ws.log.calls)` for that toolset's workspace |
| `result_chars` | `int` | `sum(c.result_chars for c in ws.log.calls)` |
| `evidence_visible` | `list[str]` | `sorted(evidence_from_log(ws.log))` |

Use a separate fresh workspace for each row. The monolithic tool is
`R.build_monolithic_tool(ws)[0].invoke({"submission_id": "interlocking-import"})`.

In [ ]:
ws_demo = R.RegistryWorkspace()
for t in R.build_registry_tools(ws_demo):
    params = list(t.args_schema.model_json_schema().get("properties", {}))
    print(f"{t.name:18s} {str(params):18s} {t.description.splitlines()[0][:80]}")

In [ ]:
def evidence_from_log(log) -> frozenset[str]:
    """Evidence kinds bought by the ok calls in a ToolCallLog (see Contract A2)."""
    # TODO
    raise NotImplementedError

def scripted_review(submission_id: str, ws) -> dict:
    tools = {t.name: t for t in R.build_registry_tools(ws)}
    """Deterministic review -> {"level", "smells", "decision"} (see Contract A2)."""
    # TODO: invoke only the tools you need, then apply reportable_smells / intake_decision
    raise NotImplementedError

control_accuracy = None   # TODO
a2 = None                 # TODO

In [ ]:
# Marking scheme for A2 -- do not edit.
with GRADER.check('A2', points=4):
    ws_c = R.RegistryWorkspace()
    out = scripted_review("signalling-equipment", ws_c)
    assert out == {"level": "taxonomy", "smells": ["no-disjointness", "subsumption-cycle"],
                   "decision": "reject"}, out
    assert not any(not c.ok for c in ws_c.log.calls), "no failed tool calls"
    assert len(ws_c.log.calls) <= 5, "call only the tools you need"
    assert evidence_from_log(ws_c.log) >= {"spectrum", "smells"}
    assert control_accuracy == 1.0
    assert isinstance(a2, pd.DataFrame) and set(a2["toolset"]) == {"decomposed", "monolithic"}
    m = a2.set_index("toolset")
    assert m.loc["monolithic", "calls"] == 1 and list(m.loc["monolithic", "evidence_visible"]) == []
    assert {"spectrum", "smells"} <= set(m.loc["decomposed", "evidence_visible"])

**Your answer:**

*(Write your answer here.)*

---

## Part B — One agent loop, or a pipeline with a verifying critic? (original Chapter 1, D1)

### B1. Agent, program, program + critic (10 pts)

The architecture question of Chapter 1, now that the course has agents. The **agent**
lets Claude choose the tool calls; the **program** fixes them (it reads the evidence
bundle `R.registry_evidence`, gathered through the same logged tools); the **critic**
checks the program's structured answer against the raw evidence with the policy as code.

1. Write `AGENT_PROMPT`: a system prompt for a tool-calling agent over
   `R.build_registry_tools(ws)` that must end its reply with one JSON object
   `{"level": ..., "smells": [...], "decision": ..., "report": ...}`; and
   `parse_triage_answer(text) -> dspy.Prediction` (fields `level`, `smells`, `decision`,
   `report`; the *last* JSON object that has a `decision` key, possibly inside a code
   fence or prose, possibly with nested brackets; empty fields when there is none).
2. Write the **program**: a signature `Triage` with input `evidence` and outputs `level`,
   `smells` (`list[str]`), `decision` and `report`, run by `dspy.Predict` with
   `A.INTAKE_POLICY` as its instruction (Chapter 1's *guidelines* program).
3. Write `critic(pred, evidence) -> (dspy.Prediction, list[str])`: a deterministic
   verification stage over the raw evidence JSON. It overrides any field the evidence
   decides — level, reportable smells, decision (`R.reportable_smells`,
   `R.intake_decision`) — keeps the report, and returns one human-readable string per
   correction.
4. On every `test` submission run (a) the agent (fresh `R.RegistryWorkspace()` each;
   `agents.build_agent(R.build_registry_tools(ws), system_prompt=AGENT_PROMPT)`,
   `agents.run_agent(agent, f"Triage the registry submission '{id}'.", log=ws.log)`;
   keep `agent_logs[id] = ws.log`), (b) the program, (c) the program followed by the
   critic. Build `b1` with columns `approach` (`"agent"`, `"program"`,
   `"program+critic"`), `id`, `score` (`R.triage_score`), `tool_calls`, `usd`
   (`agents.chat_usage(run.messages)["usd_estimate"]` for the agent, `llm.meter` for the
   program).

In writing: compare the three on score, cost and legibility. If the critic can override
all three structured fields, what is Claude for in this system — and how would you
evaluate that part?

In [ ]:
AGENT_PROMPT = "TODO"

def parse_triage_answer(text: str) -> dspy.Prediction:
    """Last JSON object with a "decision" key -> Prediction(level, smells, decision, report)."""
    # TODO
    raise NotImplementedError

# TODO: class Triage(dspy.Signature): ...; program = dspy.Predict(...)

def critic(pred, evidence: str) -> tuple[dspy.Prediction, list[str]]:
    """Override level/smells/decision from the evidence; keep the report."""
    # TODO: return (corrected prediction, list of corrections)
    raise NotImplementedError

lm = llm.configure_dspy()
agent_logs = {}
b1 = None   # TODO

In [ ]:
# Marking scheme for B1 -- do not edit.
with GRADER.check('B1', points=4):
    p = parse_triage_answer('Done.\n```json\n{"level": "taxonomy", "smells": ["missing-label"], '
                            '"decision": "revise", "report": "Label mr:X."}\n```')
    assert (p.level, p.smells, p.decision) == ("taxonomy", ["missing-label"], "revise")
    p = parse_triage_answer('{"note": {"a": 1}} then {"level": "thesaurus", "smells": [], '
                            '"decision": "accept", "report": "x", "extra": {"k": [1, 2]}}')
    assert (p.level, p.smells, p.decision) == ("thesaurus", [], "accept")
    p = parse_triage_answer("I think it should be rejected.")
    assert (p.level, p.smells, p.decision) == ("", [], "")

    evidence = R.registry_evidence("timetable-service-thesaurus")
    bad = dspy.Prediction(level="taxonomy", smells=["no-disjointness", "missing-label"],
                          decision="revise", report="keep me")
    fixed, corrections = critic(bad, evidence)
    assert (fixed.level, list(fixed.smells), fixed.decision, fixed.report) == \
        ("thesaurus", [], "accept", "keep me")
    assert len(corrections) >= 4
    good = dspy.Prediction(level="thesaurus", smells=[], decision="accept", report="ok")
    assert critic(good, evidence)[1] == []

    assert isinstance(b1, pd.DataFrame)
    assert set(b1["approach"]) == {"agent", "program", "program+critic"}
    assert {"approach", "id", "score", "tool_calls", "usd"} <= set(b1.columns)
    assert all(set(g["id"]) == {ex.id for ex in test} for _, g in b1.groupby("approach"))
    assert set(agent_logs) == {ex.id for ex in test}
    _crit = b1[b1.approach == "program+critic"].set_index("id")["score"]
    assert all(abs(_crit[ex.id] - 1.0) < 1e-9 for ex in test), "the critic decides every field"

**Your answer:**

*(Write your answer here.)*

---

## Part C — A compliance agent that proves its verdicts (original Chapter 2, D1)

---

# The formalised policy (`R.POLICY_KB`) and the audit-day facts (`R.FACTS`) are
provided. The domain is **closed**: only the named individuals exist, which turns
entailment into propositional satisfiability — decided exactly by
`R.ground_entails`. The agent's tools wrap it; your job is the agent, its output
contract, and its evaluation.

In [ ]:
ws_demo = R.PolicyWorkspace()
premises = ws_demo.premises()
gold_verdicts = {qid: R.verdict(premises, fol.parse(c)) for qid, _, c, _ in R.COMPLIANCE_QUERIES}
assert gold_verdicts == {qid: g for qid, _, _, g in R.COMPLIANCE_QUERIES}
pd.DataFrame([{"id": q, "question": text, "conclusion": c, "gold": g}
              for q, text, c, g in R.COMPLIANCE_QUERIES])

### C1. Build and evaluate the compliance agent (12 pts)

1. Write `COMPLIANCE_PROMPT`, the agent's system prompt. It must make the agent read
   the policy and facts, **check both the claim and its negation** before answering,
   answer *undetermined* when neither is entailed, and end with a JSON object
   `{"verdict": "yes" | "no" | "undetermined", "evidence": [...]}`.
2. Write `parse_verdict(text) -> str`: return the verdict from the agent's final text
   (the JSON may be inside a code fence or surrounded by prose), or `"unparseable"`.
3. For every query, build a fresh `R.PolicyWorkspace()`, its tools
   (`R.build_policy_tools(ws)`), an agent (`agents.build_agent(tools,
   system_prompt=...)`), run it with `agents.run_agent(agent, question, log=ws.log)`, and
   collect `c1` with columns `id`,
   `gold`, `predicted`, `correct`, `tool_calls`, `checked_both` (did the log contain at
   least two `check_entailment` calls?) and `usd_estimate`
   (`agents.chat_usage(run.messages)`).

In writing: report accuracy with its sample size, the cost per question, and examine
every error and every *undetermined* case. Is the Dee–R1 verdict a model error or a
gap in the policy? What would you tell the governance office?

In [ ]:
COMPLIANCE_PROMPT = """TODO"""

def parse_verdict(text: str) -> str:
    # TODO
    raise NotImplementedError

c1 = None   # TODO: run the agent on every query

In [ ]:
# Marking scheme for C1 -- do not edit.
with GRADER.check('C1', points=5):
    assert parse_verdict('{"verdict": "yes", "evidence": []}') == "yes"
    assert parse_verdict('Checked both.\n```json\n{"verdict": "undetermined", "evidence": ["x"]}\n```') == "undetermined"
    assert parse_verdict('prose {"note": 1} then {"verdict": "NO", "evidence": []}') == "no"
    assert parse_verdict("I think yes.") == "unparseable"
    assert isinstance(c1, pd.DataFrame) and len(c1) == len(R.COMPLIANCE_QUERIES)
    assert {"id", "gold", "predicted", "correct", "tool_calls", "checked_both",
            "usd_estimate"} <= set(c1.columns)
    assert set(c1["predicted"]) <= {"yes", "no", "undetermined", "unparseable"}
    assert (c1["correct"] == (c1["predicted"] == c1["gold"])).all()

**Your answer:**

*(Write your answer here.)*

---

## Grading rubric

| Part | Problem | Auto | Manual | Pts |
|---|---|---|---|---|
| A | A2 Tools, trajectories, and what a mega-tool hides (original Ch 1 A2) | 4 | 4 | 8 |
| B | B1 Agent, program, program + critic (original Ch 1 D1) | 4 | 6 | 10 |
| C | C1 Build and evaluate the compliance agent (original Ch 2 D1) | 5 | 7 | 12 |
| **Total** | | | | **30** |

*Manual* points are for the written answers and for the quality of the engineering evidence (sample sizes, cost, honest negative results).

---

## Self-test

Re-run this cell after finishing. It lists every auto-graded problem and whether its check passed.

In [ ]:
AUTO_GRADED = ['A2', 'B1', 'C1']
_ = GRADER.summary(AUTO_GRADED)